# 推理 Infra 学习实录 03：一次 generate 背后发生了什么

> 归档状态（GitHub 发布副本）：源码学习稿，未做本轮 GPU 验证。 所有 ModelScope 发布 URL 尚未知，不表示已发布。
> [配套文章](article.md) · [上一篇](../02/article.md) · [下一篇](../04/article.md)。Notebook 输出与执行计数已清空。
> 版本边界：nano-vLLM 安装与源码说明对齐已核对 commit `bb823b3e06983d71485a8e1f23715ebd87d98ef8`；已有 checkout 不自动覆盖，版本不符时应另选空目录。
> 路径说明：`/mnt/workspace` 是 ModelScope 平台缓存示例，请按实际环境调整，模型目录以 `snapshot_download()` 返回值为准。

这份 Notebook 配合本仓库文章使用，目标不是重新 benchmark，而是用一个最小 `llm.generate()` 调用，把 nano-vLLM 的源码主链路串起来。

本篇重点观察：

1. `LLM.generate()` 为什么是一个循环。
2. prompt 如何变成 `Sequence`。
3. `LLMEngine.step()` 每轮如何区分 prefill 和 decode。
4. `ModelRunner` 为什么要分别准备 prefill/decode 输入。
5. KV Cache block table 在主链路中的位置。


## 1. 环境检查

先确认 GPU、Python 和 PyTorch CUDA 状态。

In [ ]:
!nvidia-smi

In [ ]:
import sys

print("Python:", sys.version)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("Torch CUDA:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch check failed:", e)


## 2. 安装依赖

如果当前环境已经安装过，可以跳过这些单元。Notebook 中建议使用 `%pip`，这样包会安装到当前 kernel 对应的 Python 环境里。

In [ ]:
# 如未安装 ModelScope，取消注释运行
# %pip install modelscope pandas

In [ ]:
# 如未安装 nano-vLLM，取消注释运行
# %pip install git+https://github.com/GeeeekExplorer/nano-vllm.git@bb823b3e06983d71485a8e1f23715ebd87d98ef8

## 3. 下载模型

沿用前两篇使用的 `Qwen/Qwen3-0.6B`。

In [ ]:
from modelscope import snapshot_download

model_dir = snapshot_download(
    "Qwen/Qwen3-0.6B",
    cache_dir="/mnt/workspace/.cache/modelscope",
)

print("model_dir =", model_dir)


## 4. 跑一个最小 generate

这一步只是建立一个真实入口，后面读源码都围绕这行 `llm.generate(...)` 展开。

In [ ]:
from nanovllm import LLM, SamplingParams

llm = LLM(
    model_dir,
    enforce_eager=True,
    tensor_parallel_size=1,
)

sampling_params = SamplingParams(
    temperature=0.6,
    max_tokens=128,
)

prompts = ["请用三句话解释什么是大模型推理引擎。"]
outputs = llm.generate(prompts, sampling_params)

print(outputs[0]["text"])
print("output token count:", len(outputs[0]["token_ids"]))


## 5. 查看源码入口

`LLM` 本身很薄，核心逻辑在 `LLMEngine`。如果是在 Notebook 中 clone 了源码，可以用下面的命令查看。

In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path("upstream/nano-vllm")
commit = "bb823b3e06983d71485a8e1f23715ebd87d98ef8"
if not repo_dir.exists():
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "https://github.com/GeeeekExplorer/nano-vllm.git", str(repo_dir)],
        check=True,
    )
    subprocess.run(["git", "-C", str(repo_dir), "checkout", commit], check=True)
else:
    print("保留已有 checkout，不自动 pull 或覆盖：", repo_dir)
actual_commit = subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True,
).strip()
if actual_commit != commit:
    raise RuntimeError(f"源码版本不符：{actual_commit}；请另选空目录克隆固定版本，不覆盖已有工作。")
print("source commit:", actual_commit)

In [ ]:
from pathlib import Path

src_root = repo_dir / "nanovllm"  # 上一单元已核对固定 commit 的独立 checkout
source_base = "https://github.com/GeeeekExplorer/nano-vllm/blob/bb823b3e06983d71485a8e1f23715ebd87d98ef8/nanovllm"
for rel in ["llm.py", "engine/llm_engine.py"]:
    print("=" * 80)
    print(f"{source_base}/{rel}")
    path = src_root / rel
    print("".join(path.read_text().splitlines(True)[:140]))

## 6. 手动观察 tokenizer 和 Sequence

`add_request()` 的核心动作是：文本 prompt 先被 tokenizer 编码成 token ids，然后包装成 `Sequence`，再放进 scheduler。

In [ ]:
from transformers import AutoTokenizer
from nanovllm.engine.sequence import Sequence

# 如果当前环境中 nanovllm 包不可直接导入内部模块，请先确认 nano-vLLM 已安装。
tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)

prompt = "请用三句话解释什么是大模型推理引擎。"
prompt_token_ids = tokenizer.encode(prompt)
seq = Sequence(prompt_token_ids, sampling_params)

print("prompt token count:", len(prompt_token_ids))
print("seq_id:", seq.seq_id)
print("status:", seq.status)
print("num_prompt_tokens:", seq.num_prompt_tokens)
print("num_cached_tokens:", seq.num_cached_tokens)
print("num_scheduled_tokens:", seq.num_scheduled_tokens)
print("is_prefill:", seq.is_prefill)
print("block_table:", seq.block_table)


## 7. 用表格整理 Sequence 字段

推理服务里的请求不只是一段文本，而是一个持续变化的状态对象。

In [ ]:
import pandas as pd

sequence_fields = [
    ("seq_id", "请求编号"),
    ("status", "WAITING / RUNNING / FINISHED"),
    ("token_ids", "prompt token 和生成 token"),
    ("last_token", "decode 阶段输入的上一个 token"),
    ("num_prompt_tokens", "prompt token 数"),
    ("num_cached_tokens", "已写入 KV Cache 的 token 数"),
    ("num_scheduled_tokens", "当前 step 被调度的 token 数"),
    ("is_prefill", "当前是否处于 prefill 阶段"),
    ("block_table", "Sequence 到 KV Cache block 的映射"),
    ("temperature / max_tokens / ignore_eos", "采样参数"),
]

pd.DataFrame(sequence_fields, columns=["字段", "含义"])


## 8. generate 主循环伪代码

读 `LLMEngine.generate()` 时，先抓住这个结构就够了。

In [ ]:
generate_flow = """
for prompt in prompts:
    add_request(prompt, sampling_params)

while not scheduler.is_finished():
    outputs, num_tokens = step()
    if num_tokens > 0:
        update prefill throughput
    else:
        update decode throughput

return tokenizer.decode(completion_token_ids)
"""

print(generate_flow)


## 9. step 的三段式

`step()` 是推理系统的一轮执行：先调度，再执行模型，再回写状态。

In [ ]:
step_flow = """
seqs, is_prefill = scheduler.schedule()
token_ids = model_runner.run(seqs, is_prefill)
scheduler.postprocess(seqs, token_ids, is_prefill)
"""

print(step_flow)


## 10. Scheduler 状态变化

源码中 scheduler 维护 `waiting` 和 `running` 两个队列。可以先用下面的状态图理解。

In [ ]:
scheduler_flow = """
add_request
  -> WAITING, waiting queue

调度最后一段 prefill（模型执行之前）
  -> RUNNING, running queue
  -> 执行最后一段 prefill，完成后采样首 token

decode 生成 token
  -> 继续 RUNNING 或 FINISHED

KV block 不足
  -> preempt
  -> WAITING, waiting queue

FINISHED
  -> release KV blocks
"""

print(scheduler_flow)


## 11. Prefill 和 decode 的输入差异

`ModelRunner.run()` 根据 `is_prefill` 进入 `prepare_prefill()` 或 `prepare_decode()`。

In [ ]:
prefill_decode = pd.DataFrame([
    ("处理内容", "prompt 或 prompt chunk", "每个请求通常 1 个 last token"),
    ("主要元数据", "cu_seqlens_q / cu_seqlens_k / slot_mapping", "context_lens / block_tables / slot_mapping"),
    ("Attention", "flash_attn_varlen_func", "flash_attn_with_kvcache"),
    ("主要压力", "计算量", "KV Cache 读取、调度开销、显存带宽"),
], columns=["维度", "Prefill", "Decode"])

prefill_decode


## 12. KV Cache block 大小估算

`ModelRunner.allocate_kv_cache()` 中会根据可用显存估算可以分配多少 KV block。

In [ ]:
kv_formula = """
block_bytes =
    2
  * num_hidden_layers
  * block_size
  * num_kv_heads_per_rank
  * head_dim
  * dtype.itemsize
"""

print(kv_formula)


## 13. generate 主链路总结

这一篇最后要能把下面这条链路讲清楚。

In [ ]:
full_flow = """
prompt
-> tokenizer.encode
-> Sequence
-> Scheduler waiting queue
-> Scheduler.schedule
-> ModelRunner.prepare_prefill / prepare_decode
-> Attention writes or reads KV Cache
-> Sampler selects next token
-> Scheduler.postprocess
-> tokenizer.decode
-> output text
"""

print(full_flow)


## 14. 下一步

下一篇可以继续深入：

- `BlockManager.allocate()` 如何分配 KV block。
- `BlockManager.can_append()` 为什么用 `len(seq) % block_size == 1` 判断是否需要新 block。
- `hash_blocks()` 如何为 prefix cache 做准备。
- prefix cache 为什么必须要求完整前缀一致。
